# Question 5
## a. - d.

In [ ]:
import re
from collections import Counter
from itertools import chain, pairwise
from pprint import pprint

import matplotlib.pyplot as plt
from nltk.corpus import gutenberg
from tqdm import tqdm


_TOKEN_PATTERN = re.compile(r"[a-zA-Z]+|[0-9]+|[^a-zA-Z0-9\s]")

class BPETokenizer:
    """Byte Pair Encoding (BPE) Tokenizer implementation.
    Maintains internal vocabulary and merge rules learned during training.
    """

    def __init__(self):
        self.base_corpus = None
        self.corpus = None
        self.vocabulary = None
        # Ordered by insertion, maps Pair to merge index for O(1) priority checks
        self.bpe_ranks: dict[tuple[str, str], int] = {}

    def prepare_corpus(self, text: str):
        pre_tokens = self._pre_tokenize(text)
        self.base_corpus = Counter(tuple(token) for token in pre_tokens)
        self.reset_corpus()

    def reset_corpus(self):
        self.corpus = Counter(self.base_corpus)

    def train_bpe(self, n_merges: int):
        # V <- all unique characters in C
        self.vocabulary = dict.fromkeys(chain.from_iterable(self.corpus))
        self.bpe_ranks = {}  # (t_l, t_r) -> merge order index

        with tqdm(
                total=n_merges, desc="Training BPE", unit="merge", dynamic_ncols=True, disable=True
            ) as pbar:
            for step in range(n_merges):
                # t_L, t_R <- Most frequent pair of adjacent tokens in C
                pair = self._get_most_frequent_pair()

                if pair is None:
                    break

                t_l, t_r = pair
                t_new = t_l + t_r

                # Save merge rule order
                self.bpe_ranks[pair] = step

                # update the vocabulary
                self.vocabulary[t_new] = None

                # Replace each occurrence of t_L , t_R in C with t_NEW
                new_corpus = {}
                for word, freq in self.corpus.items():
                    # fast pre-filter to bypass calling _merge_tuple on words that have no chance of containing the pair
                    if t_l in word and t_r in word:
                        new_word = self._merge_tuple(word, t_l, t_r, t_new)
                        new_corpus[new_word] = new_corpus.get(new_word, 0) + freq
                    else:
                        new_corpus[word] = freq
                self.corpus = new_corpus

                pbar.update(1)
                # Display current merged token and resulting vocab size
                pbar.set_postfix(pair=f"({t_l}, {t_r})", vocab=len(self.vocabulary))

    def tokenize(self, text: str) -> list[str]:
        # Split the text using the same regex applied during training
        pre_tokens = self._pre_tokenize(text)
        result = []

        for pre_token in pre_tokens:
            # Segment into characters
            word = tuple(pre_token)

            # Run BPE encoder
            encoded_tokens = self._encode(word)

            # Run the BPE merge rules
            result.extend(encoded_tokens)

        return result

    def _encode(self, word_tuple: tuple[str, ...]) -> tuple[str, ...]:
        current_tuple = word_tuple

        # Words of length <= 1 cannot be merged further
        if len(current_tuple) < 2:
            return current_tuple

        while True:
            highest_rank = None
            highest_rank_pair = None

            # find the highest ranking pair (lowest numerical rank)
            for i in range(len(current_tuple) - 1):
                pair = (current_tuple[i], current_tuple[i + 1])
                if pair in self.bpe_ranks:
                    rank = self.bpe_ranks[pair]
                    if highest_rank is None or rank < highest_rank:
                        highest_rank = rank
                        highest_rank_pair = pair

            # No further valid merges exist
            if highest_rank_pair is None:
                break

            # Construct the new tuple by merging all occurrences of highest_rank_pair
            first, second = highest_rank_pair
            new_tuple = []
            i = 0
            while i < len(current_tuple):
                if (
                    i < len(current_tuple) - 1
                    and current_tuple[i] == first
                    and current_tuple[i + 1] == second
                ):
                    new_tuple.append(first + second)
                    i += 2
                else:
                    new_tuple.append(current_tuple[i])
                    i += 1

            current_tuple = tuple(new_tuple)

            if len(current_tuple) == 1:
                break

        return current_tuple

    def _get_most_frequent_pair(self) -> tuple[str, str] | None:
        counter: Counter[tuple[str, str]] = Counter()

        for word in self.corpus:
            counter.update(pairwise(word))

        top_pair = counter.most_common(1)

        return top_pair[0][0] if top_pair else None

    @staticmethod
    def _pre_tokenize(text: str) -> list[str]:
        return _TOKEN_PATTERN.findall(text)

    @staticmethod
    def _merge_tuple(word_tuple: tuple, t_l: str, t_r: str, merged_token: str) -> tuple[str, ...]:
        new_word = []
        i = 0
        n = len(word_tuple)
        while i < n:
          if i < n - 1 and word_tuple[i] == t_l and word_tuple[i + 1] == t_r:
            new_word.append(merged_token)
            i += 2
          else:
            new_word.append(word_tuple[i])
            i += 1
        return tuple(new_word)

## e.

In [ ]:
tokenizer = BPETokenizer()

alice_raw = gutenberg.raw('carroll-alice.txt')
tokenizer.prepare_corpus(alice_raw)
tokenizer.train_bpe(10)

all_ranks = list(tokenizer.bpe_ranks.items())

def print_ranks(header: str, ranks: list[tuple[tuple[str, str], int]]) -> None:
    print(header)
    print(f"{'Rank':<6} {'Pair':<20} {'Merged':<15}")
    print("-" * 45)

    for (left, right), rank in ranks:
        pair_str = f"('{left}', '{right}')"
        print(f"{rank:<6} {pair_str:<20} {left + right:<15}")

print_ranks("First 10 pairs:", all_ranks[:10])
print_ranks("Last 10 pairs:", all_ranks[-10:])

## f.

In [ ]:
text_1 = "Alice was beginning to get very tired."
text_2 = "The rabbit disappeared down the hole."
text_3 = "Modern language models are based on the Transformer architecture."

for text in [text_1, text_2, text_3]:
    tokenized = tokenizer.tokenize(text)
    pprint(tokenized, compact=True, width=100)



## g.

In [ ]:
n_iters = (50, 100, 200, 500, 1000)
stats = []
for n in n_iters:
    tokenizer.reset_corpus()
    tokenizer.train_bpe(n)
    n_tokens = len(tokenizer.bpe_ranks)
    n1 = len(tokenizer.tokenize(text_1))
    n2 = len(tokenizer.tokenize(text_2))
    n3 = len(tokenizer.tokenize(text_3))
    stats.append((n, n_tokens, n1, n2, n3))

# Unpack columns for plotting
iters, n_tokens, len_t1, len_t2, len_t3 = zip(*stats)

fig, (ax_vocab, ax_tokens) = plt.subplots(
    2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)

# Top Subplot: Vocabulary Growth
ax_vocab.plot(
    iters,
    n_tokens,
    marker="D",
    color="#ff7f0e",
    linewidth=2,
    label="Vocab Size",
)
ax_vocab.set_ylabel("Vocab Size")
ax_vocab.set_title("Vocabulary Size & Sentence Compression vs. Merges")
ax_vocab.grid(True, linestyle="--", alpha=0.5)
ax_vocab.legend(loc="upper left")

# Bottom Subplot: Sentence Compression
ax_tokens.plot(
    iters, len_t1, marker="o", color="#1f77b4", label="Sentence 1 Tokens"
)
ax_tokens.plot(
    iters, len_t2, marker="s", color="#2ca02c", label="Sentence 2 Tokens"
)
ax_tokens.plot(
    iters, len_t3, marker="^", color="#d62728", label="Sentence 3 Tokens"
)
ax_tokens.set_xlabel("Number of Merges ($n$)")
ax_tokens.set_ylabel("Tokens per Sentence")
ax_tokens.grid(True, linestyle="--", alpha=0.5)
ax_tokens.legend(loc="upper right")

plt.tight_layout()
plt.show()


# Discussion
- **Vocabulary size V** grows strictly linearly with the number of merges n. The corpus size S &gg; 1000, such that each iteration can extract one new token.
- **Tokens per sentence** drop rapidly in range n < 200 and then plateaue beyond n > 500. The reason is the sentence lengths are short. Also, Sentence 3 contains more than twice as many tokens as the first two sentences since it is both longer and contains words that are not likely to be in Lewis Carrol's book for children, such as "models", "Transformer", and "architecture". These words remain split into shorter subwords and individual characters, unlike the words in sentences 1 and 2.

# AI Assistance Disclaimer

* **Tool & Model:** Google Gemini (Gemini Advanced / Gemini 2.5)
* **Scope of Use:** Applied across programming tasks and theoretical sections of this assignment.
* **Nature of Assistance:** Clarification and explanation of course concepts, Python API and library documentation lookups, and code correctness/sanity checks. No AI-generated code or text was directly copied or submitted as a whole solution; all submitted code and answers were independently implemented, reviewed, and fully understood.
* **Development History:** Complete work tracking and version history for this assignment are available at [https://github.com/dgomon/nlp](https://github.com/dgomon/nlp).